# Qwen3-VL — Zero-Shot Anomaly Explainer  ·  All-in-One Colab

**Purpose**: Run the full pipeline on Colab Pro: load Qwen3-VL-32B, read RTFM outputs from **Google Drive**, generate explanations in-process (no tunnel), then GPT-4o-as-judge. Results are written back to Drive.

```
Google Drive (rtfm_pipeline_outputs/, annotations.json)
        │
        ▼  Mount Drive → load metadata → copy frames to local
  THIS notebook  (Qwen3-VL-32B on A100)
        │  In-process inference (no HTTP, no tunnel)
        ▼
  explanation_qwen_zeroshot.json, judge_qwen_zeroshot.json → back to Drive
```

### Workflow
1. **Cell 1** — GPU check (confirm A100 / enough VRAM)
2. **Cell 2** — Install dependencies (run once, then **Runtime → Restart session**)
3. **Cell 3** — Load Qwen3-VL-32B model + processor
4. **Cell 4** — Mount Drive, load RTFM metadata from Drive, copy frames to local disk
5. **Cell 5** — Define prompts + `run_qwen_inference_local` + `build_explain_request` + GPT-4o judge
6. **Cell 6** — Run Qwen on all videos, save `explanation_qwen_zeroshot.json` per video to Drive
7. **Cell 7** — GPT-4o-as-judge on each explanation, save `judge_qwen_zeroshot.json` to Drive

### Prerequisites on Drive
- `MyDrive/rtfm_pipeline_outputs/` — per-video folders with `metadata.json` and `snippet_*_frame_*.jpg`
- `MyDrive/annotations.json` — human ground-truth explanations (for judge)
- **Colab Secrets**: add `OPENAI_API_KEY` (for GPT-4o judge) and optionally `HF_TOKEN` (for gated model)

> **Runtime**: Set to **GPU → A100** (Colab Pro) for Qwen3-VL-32B-Instruct (~64 GB BF16).  
> Fallback: `Qwen/Qwen2.5-VL-7B-Instruct` fits on a T4 (~15 GB).

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────────────────────
import torch

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    name   = torch.cuda.get_device_name(0)
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    free   = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU      : {name}')
    print(f'VRAM     : {total:.1f} GB total  |  {free:.1f} GB free')
    if total < 30:
        print('⚠  < 30 GB — switch to Qwen2.5-VL-7B-Instruct in Cell 3')
    elif total < 70:
        print('✓  Fits Qwen3-VL-32B BF16 (~64 GB) — may be tight, watch memory')
    else:
        print('✓  A100-80 GB — ideal for Qwen3-VL-32B BF16')
else:
    raise RuntimeError('No GPU found. Set Runtime → Change runtime type → GPU (A100).')

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : NVIDIA A100-SXM4-80GB
VRAM     : 85.1 GB total  |  84.6 GB free
✓  A100-80 GB — ideal for Qwen3-VL-32B BF16


In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# Run ONCE.  If packages were freshly installed, do:
#   Runtime → Restart session
# then re-run from Cell 1 (no need to re-install).

import subprocess, sys, os

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'transformers>=4.45.0',
    'accelerate>=0.30.0',
    'qwen-vl-utils[decord]',
    'openai',
])

import transformers
print(f'transformers : {transformers.__version__}')
print('\nInstall done.')
print('>>> If this was the first install, do: Runtime → Restart session')
print('>>> Then re-run from Cell 1 (skip this cell).')

cloudflared installed.
transformers : 5.3.0

Install done.
>>> If this was the first install, do: Runtime → Restart session
>>> Then re-run from Cell 1 (skip this cell).


In [ ]:
# ── Cell 3: Load Qwen3-VL model ───────────────────────────────────────────────
#
# ┌─ Model options ────────────────────────────────────────────────────────────┐
# │  MODEL_ID = 'Qwen/Qwen3-VL-32B-Instruct'    BF16  ~64 GB  ← default     │
# │  MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'   BF16  ~15 GB  ← T4 fallback │
# └────────────────────────────────────────────────────────────────────────────┘

import os, gc, torch
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

# ── Optional: HuggingFace auth (needed for gated models) ─────────────────────
HF_TOKEN = os.environ.get('HF_TOKEN', '')  # ← paste token if needed, or add to Colab Secrets as HF_TOKEN
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
    except Exception:
        pass

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f'HF authenticated  ({HF_TOKEN[:8]}...{HF_TOKEN[-4:]})')
else:
    print('No HF token — downloads may be rate-limited (usually fine for Qwen).')

# ── Select model ─────────────────────────────────────────────────────────────
MODEL_ID = 'Qwen/Qwen3-VL-32B-Instruct'
# MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'   # ← T4 fallback

# ── Import correct model class (Qwen3-VL or Qwen2-VL) ────────────────────────
try:
    from transformers import Qwen3VLForConditionalGeneration as QwenVLModel
    print('Model class : Qwen3VLForConditionalGeneration')
except ImportError:
    from transformers import Qwen2VLForConditionalGeneration as QwenVLModel
    print('Model class : Qwen2VLForConditionalGeneration  (upgrade transformers for Qwen3)')

# ── Free memory, set alloc config ────────────────────────────────────────────
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()
print(f'GPU free before load : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# ── Load model ────────────────────────────────────────────────────────────────
print(f'\nLoading {MODEL_ID} ...')
model = QwenVLModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

print('\nModel loaded.')
print(f'GPU allocated : {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'GPU reserved  : {torch.cuda.memory_reserved()/1e9:.1f} GB')

HF authenticated  (hf_kTZJb...WnMI)
Model class : Qwen3VLForConditionalGeneration
GPU free before load : 84.6 GB

Loading Qwen/Qwen3-VL-32B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1058 [00:00<?, ?it/s]


Model loaded.
GPU allocated : 66.7 GB
GPU reserved  : 66.7 GB


In [ ]:
# ── Cell 4: Mount Drive + load RTFM metadata + copy frames to local ─────────────
from google.colab import drive
drive.mount('/content/drive')

import json
import shutil
from pathlib import Path

# Paths on Drive (adjust if your folder names differ)
OUTPUT_DIR       = Path('/content/drive/MyDrive/rtfm_pipeline_outputs')
ANNOTATIONS_PATH = Path('/content/drive/MyDrive/annotations.json')
LOCAL_FRAMES     = Path('/content/qwen_frames')
LOCAL_FRAMES.mkdir(parents=True, exist_ok=True)

# Load per-video metadata
video_metas = {}
for video_folder in sorted(OUTPUT_DIR.iterdir()):
    if not video_folder.is_dir() or video_folder.name.startswith('.'):
        continue
    meta_file = video_folder / 'metadata.json'
    if not meta_file.exists():
        continue
    with open(meta_file) as f:
        meta = json.load(f)
    vname = meta.get('video_id', video_folder.name)
    valid_frames = []
    for fr in meta.get('extracted_frames', []):
        fname = fr.get('file')
        if not fname:
            continue
        drive_path = video_folder / fname
        if not drive_path.exists():
            continue
        local_path = LOCAL_FRAMES / vname / fname
        local_path.parent.mkdir(parents=True, exist_ok=True)
        if not local_path.exists():
            shutil.copy2(drive_path, local_path)
        fr = dict(fr)
        fr['_abs_path'] = str(local_path)
        valid_frames.append(fr)
    if not valid_frames:
        continue
    meta = dict(meta)
    meta['video_id'] = vname
    meta['extracted_frames'] = valid_frames
    video_metas[vname] = meta

print(f'Loaded {len(video_metas)} videos from {OUTPUT_DIR}')

# Load human annotations (for judge)
annotations = {}
if ANNOTATIONS_PATH.exists():
    with open(ANNOTATIONS_PATH) as f:
        raw = json.load(f)
    annotations = {e['video_id']: e for e in raw if 'video_id' in e}
    print(f'Annotations: {len(annotations)} videos')
else:
    print(f'WARNING: {ANNOTATIONS_PATH} not found — judge will use generic GT for normal FP clips.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 46 videos from /content/drive/MyDrive/rtfm_pipeline_outputs
Annotations: 44 videos


In [ ]:
# ── Cell 5: Prompts + in-process inference + build_explain_request + judge ─────
import base64
import gc
import json
import os
import tempfile
import time
from pathlib import Path

import torch
from openai import OpenAI
from qwen_vl_utils import process_vision_info

# OpenAI API key (Colab Secrets or env)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
except Exception:
    pass
if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
assert OPENAI_API_KEY, 'Add OPENAI_API_KEY in Colab Secrets (key icon in sidebar)'
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
openai_client = OpenAI(api_key=OPENAI_API_KEY)

MAX_NEW_TOKENS = 300
SLEEP_BETWEEN = 0.5

SYSTEM_PROMPT = """You are a surveillance video anomaly analyst. You will be shown a set of \
frames sampled from a surveillance video that has been flagged as anomalous \
by a weakly-supervised anomaly detection model (RTFM).

The frames are ordered temporally. Each frame comes from a specific temporal \
snippet of the video, and you are given the anomaly score for that snippet \
(0 = normal, 1 = highly anomalous).

The frames were specifically selected from the anomalous portions of the \
video — they represent the onset, peak, and resolution of the detected anomaly.

Your task: Based on ALL the frames and their anomaly scores together, provide \
a single concise explanation (2-3 sentences) of what anomalous activity is \
happening. Focus on:
- WHAT is happening (the specific anomalous activity)
- WHO/WHAT is involved (people, vehicles, objects — describe appearance)
- WHEN in the sequence it starts and ends
- WHY it is anomalous (how it deviates from normal pedestrian behaviour)

Respond with ONLY a JSON object in this exact format:
{\"explanation\": \"...\"}"""

JUDGE_PROMPT = """You are an impartial judge evaluating the quality of an AI-generated \
explanation of an anomalous event in a surveillance video.

You will be given:
1. A HUMAN ground-truth explanation written by someone who watched the full video.
2. An AI-GENERATED explanation produced by a vision-language model that only \
saw a few sampled frames guided by anomaly scores from a weakly-supervised \
anomaly detection model (RTFM).

Score the AI explanation on these 4 criteria (each 1-5):
- correctness  : Does the AI identify the same anomaly as the human?
- specificity  : Does the AI mention specific details (objects, people, actions, location)?
- completeness : Does the AI capture all aspects the human mentioned?
- fluency      : Is the explanation well-written, clear, and natural?

Also provide a brief justification (1-2 sentences).

Respond with ONLY a JSON object:
{\"correctness\": 1-5, \"specificity\": 1-5, \"completeness\": 1-5, \"fluency\": 1-5, \"justification\": \"...\"}"""

NORMAL_FP_GT = 'There is nothing anomalous in this video. All pedestrians are walking normally.'


def run_qwen_inference_local(body: dict) -> str:
    """Run Qwen in-process; body = {system_prompt, intro_text, frames: [{label, b64}], max_new_tokens}. Returns explanation string."""
    tmp_paths = []
    try:
        content = [{"type": "text", "text": body["intro_text"]}]
        for fr in body["frames"]:
            tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
            tmp.write(base64.b64decode(fr["b64"]))
            tmp.close()
            tmp_paths.append(tmp.name)
            content.append({"type": "text", "text": fr["label"]})
            content.append({"type": "image", "image": f"file://{tmp.name}"})
        messages = [
            {"role": "system", "content": body["system_prompt"]},
            {"role": "user", "content": content},
        ]
        text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)[:2]
        inputs = processor(text=[text_input], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=body.get("max_new_tokens", MAX_NEW_TOKENS), do_sample=False)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
        raw = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        text = raw
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        try:
            parsed = json.loads(text.strip())
        except json.JSONDecodeError:
            parsed = {"explanation": raw}
        return parsed.get("explanation", raw)
    except Exception as e:
        return f"ERROR: {e}"
    finally:
        for p in tmp_paths:
            try:
                os.unlink(p)
            except Exception:
                pass
        gc.collect()


def build_explain_request(meta: dict) -> dict:
    """Build body dict for run_qwen_inference_local."""
    frames = meta["extracted_frames"]
    segs = meta.get("anomalous_segments", [])

    def seg_label(s):
        start = s.get("start_snippet", s.get("start", "?"))
        end = s.get("end_snippet", s.get("end", "?"))
        return f"snippets {start}-{end}"

    seg_str = ", ".join(seg_label(s) for s in segs) if segs else "unknown"
    score_str = ", ".join(f"snippet {f['snippet_idx']}={f['score']:.3f}" for f in frames)
    intro = (
        f"This video has {meta['n_segments']} temporal snippets (~16 frames each). "
        f"RTFM flagged these contiguous segments as anomalous: [{seg_str}]. "
        f"Video-level anomaly gate score: {meta['gate_score']:.3f} "
        f"(0=normal, 1=highly anomalous; threshold=0.2). "
        f"Below are {len(frames)} frames sampled from the anomalous segments. "
        f"Per-snippet anomaly scores: [{score_str}]."
    )
    frame_items = []
    for fr in frames:
        img_path = fr.get("_abs_path", "")
        if not Path(img_path).exists():
            continue
        with open(img_path, "rb") as fh:
            b64 = base64.b64encode(fh.read()).decode("utf-8")
        frame_items.append({
            "label": f"Frame from snippet {fr['snippet_idx']} (frame #{fr['frame_num']}, anomaly score: {fr['score']:.3f}):",
            "b64": b64,
        })
    return {"system_prompt": SYSTEM_PROMPT, "intro_text": intro, "frames": frame_items, "max_new_tokens": MAX_NEW_TOKENS}


def call_gpt4o_judge(human_expl: str, ai_expl: str) -> dict:
    user_msg = f'HUMAN ground-truth explanation:\n"{human_expl}"\n\nAI-generated explanation:\n"{ai_expl}"'
    try:
        resp = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "system", "content": JUDGE_PROMPT}, {"role": "user", "content": user_msg}],
            max_tokens=300,
            response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        return {"correctness": None, "specificity": None, "completeness": None, "fluency": None, "justification": f"ERROR: {e}"}


print("Prompts and helpers ready.")

Prompts and helpers ready.


In [ ]:
# ── Cell 6: Run Qwen on all videos, save explanation_qwen_zeroshot.json to Drive ─
qwen_explanations = {}
n_total = len(video_metas)

for i, (vname, meta) in enumerate(sorted(video_metas.items())):
    n_frames = len(meta["extracted_frames"])
    gate = meta["gate_score"]
    print(f"[{i+1}/{n_total}] {vname}  gate={gate:.3f}  frames={n_frames} ...", end=" ", flush=True)

    body = build_explain_request(meta)
    n_sent = len(body["frames"])
    explanation = run_qwen_inference_local(body)

    short = f'"{explanation[:80]}..."' if len(explanation) > 80 else f'"{explanation}"'
    print(f"-> {short}")

    result = {
        "video_id": vname,
        "model": MODEL_ID,
        "gate_score": gate,
        "n_segments": meta["n_segments"],
        "anomalous_segments": meta.get("anomalous_segments", []),
        "n_frames_sent": n_sent,
        "explanation": explanation,
    }
    qwen_explanations[vname] = result

    vid_out = OUTPUT_DIR / vname
    vid_out.mkdir(parents=True, exist_ok=True)
    with open(vid_out / "explanation_qwen_zeroshot.json", "w") as f:
        json.dump(result, f, indent=2)

with open(OUTPUT_DIR / "qwen_zeroshot_explanations_summary.json", "w") as f:
    json.dump(list(qwen_explanations.values()), f, indent=2)

n_ok = sum(1 for v in qwen_explanations.values() if not str(v["explanation"]).startswith("ERROR"))
print(f"\nDone. {n_ok} OK  |  {len(qwen_explanations) - n_ok} errors")
print(f"Saved to {OUTPUT_DIR}")

[1/46] 01_0015  gate=0.664  frames=4 ... 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


-> "An anomalous event occurs between snippets 12 and 19, where a person in a patter..."
[2/46] 01_0025  gate=1.000  frames=8 ... -> "An anomalous event occurs when a person on a bicycle enters the pedestrian plaza..."
[3/46] 01_0027  gate=0.646  frames=8 ... -> "A group of young boys is seen running and moving erratically across a paved plaz..."
[4/46] 01_0028  gate=0.409  frames=7 ... -> "Anomalous activity occurs in two separate time segments: first from snippet 5 to..."
[5/46] 01_0051  gate=0.959  frames=8 ... -> "A man on a bicycle enters the paved plaza from the left, riding across the open ..."
[6/46] 01_0052  gate=0.776  frames=7 ... -> "An anomalous event occurs between snippets 4 and 17, where a person on a bicycle..."
[7/46] 01_0053  gate=0.998  frames=8 ... -> "A green three-wheeled utility vehicle, driven by a person in a red shirt, enters..."
[8/46] 01_0056  gate=0.617  frames=4 ... -> "An anomalous event occurs between snippets 20 and 26, where a person on a skateb..."
[

In [ ]:
# ── Cell 7: GPT-4o-as-judge; save judge_qwen_zeroshot.json to Drive ─────────────
qwen_judge_results = []
n_anom, n_fp = 0, 0

for i, (vname, expl_data) in enumerate(sorted(qwen_explanations.items())):
    ai_expl = expl_data.get("explanation", "")
    if not ai_expl or str(ai_expl).startswith("ERROR"):
        print(f"  SKIP {vname}: no valid explanation")
        continue

    clip_id = vname.split("_")[1] if "_" in vname else vname
    is_truly_anomalous = len(clip_id) == 4

    if is_truly_anomalous and vname in annotations:
        human_expl = annotations[vname]["explanation"]
        gt_start = annotations[vname].get("anomaly_start_frame")
        gt_end = annotations[vname].get("anomaly_end_frame")
        video_type = "anomalous"
        n_anom += 1
    else:
        human_expl = NORMAL_FP_GT
        gt_start = gt_end = None
        video_type = "normal_FP"
        n_fp += 1

    print(f"[{i+1}] {vname}  [{video_type}]")
    scores = call_gpt4o_judge(human_expl, ai_expl)
    print(f"  -> C={scores.get('correctness')} S={scores.get('specificity')} Co={scores.get('completeness')} F={scores.get('fluency')}")

    result = {
        "video_id": vname,
        "model": MODEL_ID,
        "video_type": video_type,
        "human_explanation": human_expl,
        "ai_explanation": ai_expl,
        "gate_score": expl_data.get("gate_score"),
        "gt_anomaly_start": gt_start,
        "gt_anomaly_end": gt_end,
        "scores": scores,
    }
    qwen_judge_results.append(result)

    vid_out = OUTPUT_DIR / vname
    with open(vid_out / "judge_qwen_zeroshot.json", "w") as f:
        json.dump(result, f, indent=2)

    time.sleep(SLEEP_BETWEEN)

with open(OUTPUT_DIR / "qwen_zeroshot_judge_summary.json", "w") as f:
    json.dump(qwen_judge_results, f, indent=2)

print(f"\nJudged {len(qwen_judge_results)} videos  ({n_anom} anomalous + {n_fp} normal FP)")
print(f"Saved to {OUTPUT_DIR}")

[1] 01_0015  [anomalous]
  -> C=2 S=3 Co=2 F=4
[2] 01_0025  [anomalous]
  -> C=5 S=4 Co=5 F=5
[3] 01_0027  [anomalous]
  -> C=2 S=4 Co=2 F=4
[4] 01_0028  [anomalous]
  -> C=2 S=3 Co=2 F=4
[5] 01_0051  [anomalous]
  -> C=5 S=5 Co=5 F=5
[6] 01_0052  [anomalous]
  -> C=5 S=4 Co=5 F=5
[7] 01_0053  [anomalous]
  -> C=5 S=5 Co=5 F=5
[8] 01_0056  [anomalous]
  -> C=2 S=4 Co=2 F=5
[9] 01_0135  [anomalous]
  -> C=5 S=5 Co=5 F=5
[10] 01_0136  [anomalous]
  -> C=5 S=4 Co=5 F=5
[11] 01_0139  [anomalous]
  -> C=5 S=4 Co=5 F=5
[12] 01_0162  [anomalous]
  -> C=5 S=4 Co=5 F=5
[13] 01_0177  [anomalous]
  -> C=5 S=5 Co=5 F=5
[14] 01_074  [normal_FP]
  -> C=1 S=3 Co=1 F=4
[15] 02_0161  [anomalous]
  -> C=2 S=4 Co=2 F=5
[16] 03_0031  [anomalous]
  -> C=1 S=4 Co=1 F=4
[17] 03_0032  [anomalous]
  -> C=3 S=3 Co=3 F=4
[18] 03_0033  [anomalous]
  -> C=2 S=3 Co=3 F=4
[19] 03_0059  [anomalous]
  -> C=3 S=4 Co=3 F=5
[20] 04_0004  [anomalous]
  -> C=4 S=3 Co=3 F=4
[21] 04_0011  [anomalous]
  -> C=4 S=5 Co=4 F=5
[2